# 02 Preprocess Data
This notebook will preprocess the downloaded datasets from `01_fetch_data.ipynb` and output structured parquets.

Each dataset (`_yellow`, `_green`, `_fhvhv`, `_ridership`, `_alerts`, `_weather`) will undergo the following pipe:
1. `sdf_dataset`: initial read of .parquet or .csv as a pyspark parquet
2. `sdf_dataset_clean`: parquet after removing rows with data anomalies
3. `sdf_dataset_selected`: parquet after removing columns unused by downstream analysis

Note, `_fhvhv` is large, so it is saved with a monthly for-loop. All other datasets are saved in one go.

In [1]:
from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "America/New_York") # timezone set to NYC
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR") # silence warnings

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/15 02:56:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/15 02:56:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
from functools import reduce
from operator import and_
from pathlib import Path
from pyspark.sql import functions as F

# output directory
processed_output_dir = "../data/processed"

# specify time range of data
YEARS = [2025, 2026]
MONTHS = range(1, 13)

# Analysis time window: Jan 2025 to May 2026 inclusive
START_TIME = "2025-01-01 00:00:00"
END_TIME = "2026-06-01 00:00:00"   # exclusive

## Preprocess TLC Trip Record Data (Yellow)
We want to retain all rows with data anomaly, such as the following: 
- trips with `trip_distance` <= 0
- trips with `total_amount` == 0
- trips with `RatecodeID` == 99 (Null/unknown)
- trips with `payment_type` == 6 (Voided trip)
- trips with `tpep_dropoff_datetime` - `tpep_pickup_datetime` duration < 60 seconds or > 8 hours
- trips with < 0 for any money related columns
- trips with NULL in any of `tpep_pickup_datetime`, `tpep_dropoff_datetime`, `passenger_count`, `trip_distance`, `PULocationID`, `DOLocationID`, `trip_distance`, `total_amount`

In [3]:
# Read the parquet from directory (all months combined)
sdf_yellow = spark.read.parquet('../data/raw/tlc_trips/yellow')

# Print schema for reference
sdf_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [4]:
# Remove rows with data anomalies
required_money_cols = ["fare_amount", "mta_tax", "extra", "improvement_surcharge", "tip_amount", "tolls_amount", "total_amount"]
nullable_money_cols = ["congestion_surcharge", "Airport_fee", "cbd_congestion_fee"]

sdf_yellow_clean = (
    sdf_yellow
    .withColumn( # calculate trip duration
        "trip_time",
        F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
    )
    .filter(
        (F.col("passenger_count").between(1, 5)) &                                         # passenger_count must be valid
        (F.col("trip_distance") > 0) &                                                     # trip_distance must be positive
        (F.col("total_amount") > 0) &                                                      # total_amount must be positive
        (F.col("RatecodeID").between(1, 6)) &                                              # RatecodeID must not be Null/unknown
        (F.col("payment_type").between(1, 5)) &                                            # payment_type must not be Voided trip
        (F.col("trip_time").between(60, 8 * 60 * 60)) &                                    # duration must be between 60 seconds to 8 hours
        (F.col("tpep_pickup_datetime") >= F.lit(START_TIME)) &                             # trip must begin within validated time range
        (F.col("tpep_pickup_datetime") < F.lit(END_TIME)) &                                # trip must begin within validated time range
        F.col("tpep_pickup_datetime").isNotNull() &                                        # tpep_pickup_datetime must not be NULL
        F.col("tpep_dropoff_datetime").isNotNull() &                                       # tpep_dropoff_datetime must not be NULL
        F.col("PULocationID").isNotNull() &                                                # PULocationID must not be Null
        F.col("DOLocationID").isNotNull() &                                                # DOLocationID must not be Null
        reduce(and_, [F.col(c) >= 0 for c in required_money_cols]) &                       # These columns must be non-negative and not NULL
        reduce(and_, [F.col(c).isNull() | (F.col(c) >= 0) for c in nullable_money_cols])   # These columns must be non-negative, can be NULL
    )
)

n_raw = sdf_yellow.count()
n_clean = sdf_yellow_clean.count()

print(f"Raw rows:   {n_raw:,}")
print(f"Clean rows: {n_clean:,}")
print(f"Removed:    {n_raw - n_clean:,} ({(n_raw - n_clean) / n_raw:.2%})")

Raw rows:   67,721,884
Clean rows: 46,955,438
Removed:    20,766,446 (30.66%)


In [5]:
# Remove columns unused in downstream analysis
sdf_yellow_selected = (
    sdf_yellow_clean
    .select(
        F.col("tpep_pickup_datetime").alias("pickup_datetime"),
        F.col("tpep_dropoff_datetime").alias("dropoff_datetime"),
        F.col("PULocationID").cast("int"),
        F.col("DOLocationID").cast("int"),
        (F.col("total_amount") - F.col("tip_amount")).cast("decimal(12,2)").alias("subtotal"), # passenger payment total excluding tips, to 2 d.p.
        F.lit("yellow").alias("service_type"),
    )
)

sdf_yellow_selected.write.mode("overwrite").parquet(f"{processed_output_dir}/tlc_trips/yellow")

Specifying dataset shape:
- `sdf_yellow`, raw parquet file, contains 67721884 rows x 20 columns
- `sdf_yellow_clean`, filtered for rows with valid data, contains 46955469 rows x 20 columns 
- `sdf_yellow_selected`, filtered for columns to be used downstream, contains 46955469 rows x 6 columns

---

## Preprocess TLC Trip Record Data (Green)
We want to remove all rows with data anomaly, such as the following: 
- trips with `trip_distance` <= 0
- trips with `total_amount` == 0
- trips with `RatecodeID` == 99 (Null/unknown)
- trips with `payment_type` == 6 (Voided trip)
- trips with `lpep_dropoff_datetime` - `lpep_pickup_datetime` duration < 60 seconds or > 8 hours
- trips with < 0 for any money related columns
- trips with NULL in any of `lpep_pickup_datetime`, `lpep_dropoff_datetime`, `passenger_count`, `trip_distance`, `PULocationID`, `DOLocationID`, `trip_distance`, `total_amount`

Further, since green taxi trips include street-hail (`trip_type`==1) and dispatch (`trip_type`==2), we remove where `trip_type`==2 so the green taxi cohort can be combined with the yellow taxi cohort as street-hail services, in contrast to the dispath only services provided in FHVHV.

In [6]:
# Read the parquet from directory (all months combined)
sdf_green = spark.read.parquet('../data/raw/tlc_trips/green')

# Print schema for reference
sdf_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_type: long (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [7]:
# Remove rows with data anomalies
required_money_cols = ["fare_amount", "mta_tax", "extra", "improvement_surcharge", "tip_amount", "tolls_amount", "total_amount"]
nullable_money_cols = ["congestion_surcharge", "cbd_congestion_fee"]

sdf_green_clean = (
    sdf_green
    .withColumn( # calculate trip duration
        "trip_time",
        F.unix_timestamp("lpep_dropoff_datetime") - F.unix_timestamp("lpep_pickup_datetime")
    )
    .filter(
        (F.col("trip_type") == 1) &                                                        # trip_type must be street-hail
        (F.col("passenger_count").between(1, 5)) &                                         # passenger_count must be valid
        (F.col("trip_distance") > 0) &                                                     # trip_distance must be positive
        (F.col("total_amount") > 0) &                                                      # total_amount must be positive
        (F.col("RatecodeID").between(1, 6)) &                                              # RatecodeID must not be Null/unknown
        (F.col("payment_type").between(1, 5)) &                                            # payment_type must not be Voided trip
        (F.col("trip_time").between(60, 8 * 60 * 60)) &                                    # trip_time must be between 60 seconds to 8 hours
        (F.col("lpep_pickup_datetime") >= F.lit(START_TIME)) &                             # trip must begin within validated time range
        (F.col("lpep_pickup_datetime") < F.lit(END_TIME)) &                                # trip must begin within validated time range
        F.col("lpep_pickup_datetime").isNotNull() &                                        # lpep_pickup_datetime must not be NULL
        F.col("lpep_dropoff_datetime").isNotNull() &                                       # lpep_dropoff_datetime must not be NULL
        F.col("PULocationID").isNotNull() &                                                # PULocationID must not be Null
        F.col("DOLocationID").isNotNull() &                                                # DOLocationID must not be Null
        reduce(and_, [F.col(c) >= 0 for c in required_money_cols]) &                       # These columns must be non-negative and not NULL
        reduce(and_, [F.col(c).isNull() | (F.col(c) >= 0) for c in nullable_money_cols])   # These columns must be non-negative, can be NULL
    )
)
n_raw = sdf_green.count()
n_clean = sdf_green_clean.count()

print(f"Raw rows:   {n_raw:,}")
print(f"Clean rows: {n_clean:,}")
print(f"Removed:    {n_raw - n_clean:,} ({(n_raw - n_clean) / n_raw:.2%})")

Raw rows:   802,387
Clean rows: 637,282
Removed:    165,105 (20.58%)


In [8]:
# Remove columns unused in downstream analysis
sdf_green_selected = (
    sdf_green_clean
    .select(
        F.col("lpep_pickup_datetime").alias("pickup_datetime"),
        F.col("lpep_dropoff_datetime").alias("dropoff_datetime"),
        F.col("PULocationID").cast("int"),
        F.col("DOLocationID").cast("int"),
        (F.col("total_amount") - F.col("tip_amount")).cast("decimal(12,2)").alias("subtotal"),  # passenger payment total excluding tips, to 2 d.p.
        F.lit("green").alias("service_type"),
    )
)

sdf_green_selected.write.mode("overwrite").parquet(f"{processed_output_dir}/tlc_trips/green")

Specifying dataset shape:
- `sdf_green`, raw parquet file, contains 802387 rows x 20 columns
- `sdf_green_clean`, filtered for rows with valid data, contains 637287 rows x 20 columns 
- `sdf_green_selected`, filtered for columns to be used downstream, contains 637287 rows x 6 columns

---

## Preprocess TLC Trip Record Data (FHVHV)
We want to remove all rows with data anomaly, such as the following: 
- trips with `trip_miles` <= 0
- trips with `total_amount` == 0
- trips with `RatecodeID` == 99 (Null/unknown)
- trips with `payment_type` == 6 (Voided trip)
- trips with `trip_time` duration < 60 seconds or > 8 hours
- trips with < 0 for any money related columns
- trips with NULL in any of `pickup_datetime`, `dropoff_datetime`, `passenger_count`, `trip_distance`, `PULocationID`, `DOLocationID`, `trip_distance`, `total_amount`

In [ ]:
# Read the parquet from directory (all months combined)
sdf_fhvhv = spark.read.parquet('../data/raw/tlc_trips/fhvhv')

# Print schema for reference
sdf_fhvhv.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

In [4]:
# Set directories
raw_fhvhv_dir = Path("../data/raw/tlc_trips/fhvhv")
processed_fhvhv_dir = Path("../data/processed/tlc_trips/fhvhv")
n_raw = 0
n_processed = 0

# Process each month separately to avoid spark crash
for year in YEARS:
    for month in MONTHS:
        if (year == 2026) and (month >= 6):
            continue

        input_path = raw_fhvhv_dir / f"{year}-{month:02d}.parquet"
        output_path = processed_fhvhv_dir / f"{year}-{month:02d}.parquet"

        if output_path.exists():
            print(f"Skipping FHVHV {year}-{month:02d}")
            continue
        print(f"Processing FHVHV {year}-{month:02d}")

        # Read the monthly parquet from directory
        sdf_fhvhv_month = spark.read.parquet(str(input_path))
        n_raw_month = sdf_fhvhv_month.count()
        n_raw += n_raw_month

        required_money_cols = ["base_passenger_fare", "tolls", "bcf", "sales_tax", "tips", "driver_pay"]
        nullable_money_cols = ["congestion_surcharge","airport_fee","cbd_congestion_fee"]

        sdf_fhvhv_processed = (
            sdf_fhvhv_month
            # Remove rows with data anomalies
            .filter(
                F.col("pickup_datetime").isNotNull() &                                             # pickup_datetime must not be Null
                F.col("dropoff_datetime").isNotNull() &                                            # dropoff_datetime must not be Null
                F.col("PULocationID").isNotNull() &                                                # PULocationID must not be Null
                F.col("DOLocationID").isNotNull() &                                                # DOLocationID must not be Null
                F.col("request_datetime").isNotNull() &                                            # request_datetime must not be Null
                (F.col("pickup_datetime") >= F.col("request_datetime")) &                          # request must be before pickup
                (F.col("request_datetime") >= F.lit(START_TIME)) &                                 # request time must be within specified time range
                (F.col("request_datetime") < F.lit(END_TIME)) &                                    # request time must be within specified time range
                (F.unix_timestamp("pickup_datetime") - F.unix_timestamp("request_datetime")
                ).between(0, 60 * 60) &                                                            # wait time must be between 0 seconds and 1 hour
                (F.col("dropoff_datetime") > F.col("pickup_datetime")) &                           # pickup_datetime must be before dropoff_datetime
                (F.col("trip_miles") > 0) &                                                        # trip_miles must be positive
                (F.col("trip_time").between(60, 8 * 60 * 60)) &                                    # trip_time must be between 60 seconds to 8 hours
                reduce(and_, [F.col(c) >= 0 for c in required_money_cols]) &                       # These columns must be non-negative and not NULL
                reduce(and_, [F.col(c).isNull() | (F.col(c) >= 0) for c in nullable_money_cols])   # These columns must be non-negative, can be Null
            )
            # Remove columns unused in downstream analysis
            .select(
                F.col("request_datetime"),
                F.col("pickup_datetime"),
                F.col("dropoff_datetime"),
                F.col("PULocationID").cast("int"),
                F.col("DOLocationID").cast("int"),
                (F.unix_timestamp("pickup_datetime") - F.unix_timestamp("request_datetime")
                ).cast("long").alias("wait_time"),                                                 # request-to-pickup wait time in seconds
                (
                    F.coalesce(F.col("base_passenger_fare"), F.lit(0))
                    + F.coalesce(F.col("tolls"), F.lit(0))
                    + F.coalesce(F.col("bcf"), F.lit(0))
                    + F.coalesce(F.col("sales_tax"), F.lit(0))
                    + F.coalesce(F.col("congestion_surcharge"), F.lit(0))
                    + F.coalesce(F.col("airport_fee"), F.lit(0))
                    + F.coalesce(F.col("cbd_congestion_fee"), F.lit(0))
                ).cast("decimal(12,2)").alias("subtotal"),                                         # passenger payment total excluding tips
                F.lit("fhvhv").alias("service_type"),
            )
        )

    
        (
            sdf_fhvhv_processed
            .write
            .mode("overwrite")
            .option("parquet.enable.dictionary", "false")
            .parquet(str(output_path))
        )
        
        n_processed_month = spark.read.parquet(str(output_path)).count()
        n_processed += n_processed_month
        print(
            f"{year}-{month:02d}: raw={n_raw_month:,}, "
            f"processed={n_processed_month:,}, "
            f"removed={n_raw_month - n_processed_month:,}"
            f"Saved {output_path}"
        )

        spark.catalog.clearCache()
        del sdf_fhvhv_month
        del sdf_fhvhv_processed

Processing FHVHV 2025-01


2025-01: raw=20,405,666, processed=20,197,866, removed=207,800Saved ../data/processed/tlc_trips/fhvhv/2025-01.parquet
Processing FHVHV 2025-02


2025-02: raw=19,339,461, processed=19,141,339, removed=198,122Saved ../data/processed/tlc_trips/fhvhv/2025-02.parquet
Processing FHVHV 2025-03


[128.140s][warning][gc,alloc] Executor task launch worker for task 6.0 in stage 23.0 (TID 75): Retried waiting for GCLocker too often allocating 3 words
[128.140s][warning][gc,alloc] Executor task launch worker for task 3.0 in stage 23.0 (TID 72): Retried waiting for GCLocker too often allocating 3 words
[128.141s][warning][gc,alloc] Executor task launch worker for task 7.0 in stage 23.0 (TID 76): Retried waiting for GCLocker too often allocating 3 words
[128.141s][warning][gc,alloc] Executor task launch worker for task 2.0 in stage 23.0 (TID 71): Retried waiting for GCLocker too often allocating 3 words
[128.143s][warning][gc,alloc] Executor task launch worker for task 1.0 in stage 23.0 (TID 70): Retried waiting for GCLocker too often allocating 3 words


26/08/15 02:13:20 ERROR Utils: Aborting task                        (1 + 7) / 8]
org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to file:/Users/hannahluo/Library/Mobile Documents/com~apple~CloudDocs/Documents/University/Classes resources/2026 Semester 2/MAST30034 Applied Data Science/MAST30034 Project 1 Individual/MAST30034 Project 1 Repository/data/processed/tlc_trips/fhvhv/2025-03.parquet. SQLSTATE: 58030
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:814)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.enrichWriteError(FileFormatDataWriter.scala:92)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.commit(FileFormatDataWriter.scala:123)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$executeTask$1(FileFormatWriter.scala:407)
	at org.apache.spark.util.Utils$.tryWithSafeFinallyAndFailureCallbacks(Utils.scala:1337)
	at org

ConnectionRefusedError: [Errno 61] Connection refused

In [ ]:
# # Set directories
# raw_fhvhv_dir = Path("../data/raw/tlc_trips/fhvhv")
# processed_fhvhv_dir = Path("../data/processed/tlc_trips/fhvhv")
# n_raw = 0
# n_processed = 0

# # Process each month separately to avoid spark crash
# for year in YEARS:
#     for month in MONTHS:
#         if (year == 2026) and (month >= 6):
#             continue

#         input_path = raw_fhvhv_dir / f"{year}-{month:02d}.parquet"
#         output_path = processed_fhvhv_dir / f"{year}-{month:02d}.parquet"

#         if output_path.exists():
#             print(f"Skipping FHVHV {year}-{month:02d}")
#             continue
#         print(f"Processing FHVHV {year}-{month:02d}")

#         # Read the monthly parquet from directory
#         sdf_fhvhv_month = spark.read.parquet(str(input_path))

#         required_money_cols = ["base_passenger_fare", "tolls", "bcf", "sales_tax", "tips", "driver_pay"]
#         nullable_money_cols = ["congestion_surcharge","airport_fee","cbd_congestion_fee"]

#         sdf_fhvhv_processed = (
#             sdf_fhvhv_month
#             # Remove rows with data anomalies
#             .filter(
#                 F.col("pickup_datetime").isNotNull() &                                             # pickup_datetime must not be Null
#                 F.col("dropoff_datetime").isNotNull() &                                            # dropoff_datetime must not be Null
#                 F.col("PULocationID").isNotNull() &                                                # PULocationID must not be Null
#                 F.col("DOLocationID").isNotNull() &                                                # DOLocationID must not be Null
#                 F.col("request_datetime").isNotNull() &                                            # request_datetime must not be Null
#                 (F.col("pickup_datetime") >= F.col("request_datetime")) &                          # request must be before pickup
#                 (F.col("request_datetime") >= F.lit(START_TIME)) &                                 # request time must be within specified time range
#                 (F.col("request_datetime") < F.lit(END_TIME)) &                                    # request time must be within specified time range
#                 (F.unix_timestamp("pickup_datetime") - F.unix_timestamp("request_datetime")
#                 ).between(0, 60 * 60) &                                                            # wait time must be between 0 seconds and 1 hour
#                 (F.col("dropoff_datetime") > F.col("pickup_datetime")) &                           # pickup_datetime must be before dropoff_datetime
#                 (F.col("trip_miles") > 0) &                                                        # trip_miles must be positive
#                 (F.col("trip_time").between(60, 8 * 60 * 60)) &                                    # trip_time must be between 60 seconds to 8 hours
#                 reduce(and_, [F.col(c) >= 0 for c in required_money_cols]) &                       # These columns must be non-negative and not NULL
#                 reduce(and_, [F.col(c).isNull() | (F.col(c) >= 0) for c in nullable_money_cols])   # These columns must be non-negative, can be Null
#             )
#             # Remove columns unused in downstream analysis
#             .select(
#                 F.col("request_datetime"),
#                 F.col("pickup_datetime"),
#                 F.col("dropoff_datetime"),
#                 F.col("PULocationID"),
#                 F.col("DOLocationID"),
#                 (F.unix_timestamp("pickup_datetime") - F.unix_timestamp("request_datetime")
#                 ).alias("wait_time"),                                                              # request-to-pickup wait time in seconds
#                 (
#                     F.coalesce(F.col("base_passenger_fare"), F.lit(0))
#                     + F.coalesce(F.col("tolls"), F.lit(0))
#                     + F.coalesce(F.col("bcf"), F.lit(0))
#                     + F.coalesce(F.col("sales_tax"), F.lit(0))
#                     + F.coalesce(F.col("congestion_surcharge"), F.lit(0))
#                     + F.coalesce(F.col("airport_fee"), F.lit(0))
#                     + F.coalesce(F.col("cbd_congestion_fee"), F.lit(0))
#                 ).cast("decimal(12,2)").alias("subtotal"),                                         # passenger payment total excluding tips
#                 F.lit("fhvhv").alias("service_type"),
#             )
#         )

#         n_raw_month = sdf_fhvhv_month.count()
#         n_raw += n_raw_month
#         n_processed_month = sdf_fhvhv_processed.count()
#         n_processed += n_processed_month

#         (
#             sdf_fhvhv_processed
#             .write
#             .mode("overwrite")
#             .option("parquet.enable.dictionary", "false")
#             .parquet(str(output_path))
#         )


#         print(f"Saved {output_path}")

#         spark.catalog.clearCache()
#         del sdf_fhvhv_month
#         del sdf_fhvhv_processed

Processing FHVHV 2025-01


Saved ../data/processed/tlc_trips/fhvhv/2025-01.parquet
Processing FHVHV 2025-02


Saved ../data/processed/tlc_trips/fhvhv/2025-02.parquet
Processing FHVHV 2025-03


Saved ../data/processed/tlc_trips/fhvhv/2025-03.parquet
Processing FHVHV 2025-04


Saved ../data/processed/tlc_trips/fhvhv/2025-04.parquet
Processing FHVHV 2025-05


...

Saved ../data/processed/tlc_trips/fhvhv/2026-01.parquet
Processing FHVHV 2026-02


Saved ../data/processed/tlc_trips/fhvhv/2026-02.parquet
Processing FHVHV 2026-03


Saved ../data/processed/tlc_trips/fhvhv/2026-03.parquet
Processing FHVHV 2026-04


Saved ../data/processed/tlc_trips/fhvhv/2026-04.parquet
Processing FHVHV 2026-05


Saved ../data/processed/tlc_trips/fhvhv/2026-05.parquet


In [ ]:
# Counts will only work for the file saving run which saved all at once (counted whilst saving)
print(f"Raw rows:   {n_raw:,}")
print(f"Clean rows: {n_processed:,}")
print(f"Removed:    {n_raw - n_processed:,} ({(n_raw - n_processed) / n_raw:.2%})")

Raw rows:   189,666,134
Clean rows: 187,109,572
Removed:    2,556,562 (1.35%)


In [ ]:
# Counts will only work for the file saving run which saved all at once (counted whilst saving)
print(f"Raw rows:   {n_raw:,}")
print(f"Clean rows: {n_processed:,}")
print(f"Removed:    {n_raw - n_processed:,} ({(n_raw - n_processed) / n_raw:.2%})")

Raw rows:   208,937,595
Clean rows: 206,152,360
Removed:    2,785,235 (1.33%)


Specifying dataset shape:
- `sdf_fhvhv`, raw parquet file, contains 349585798 rows x 20 columns
- `sdf_fhvhv_clean`, filtered for rows with valid data, contains 349446182 rows x 20 columns 
- `sdf_fhvhv_selected`, filtered for columns to be used downstream, contains 349446182 rows x 7 columns

---

## Preprocess MTA Subway Hourly Ridership Data
We want to remove all rows with data anomaly, such as the following: 
- data with `ridership` <= 0
- data with NULL in any of `transit_timestamp`, `station_complex_id`, `station_complex`, `latitude`, `longtitude`, `ridership`

In [3]:
# Read the CSV from directory (all months combined)
sdf_ridership = (
    spark.read
    .option("header", True)
    .csv("../data/raw/mta_ridership/*.csv")
    .select(
        F.to_timestamp("transit_timestamp").alias("transit_timestamp"),
        F.col("station_complex_id").cast("string").alias("station_complex_id"),
        F.col("station_complex").cast("string").alias("station_complex"),
        F.col("borough").cast("string").alias("borough"),
        F.col("latitude").cast("double").alias("latitude"),
        F.col("longitude").cast("double").alias("longitude"),
        F.col("ridership").cast("double").alias("ridership"),
        F.col("transfers").cast("double").alias("transfers")
    )
)

# Print schema for reference
sdf_ridership.printSchema()

root
 |-- transit_timestamp: timestamp (nullable = true)
 |-- station_complex_id: string (nullable = true)
 |-- station_complex: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- ridership: double (nullable = true)
 |-- transfers: double (nullable = true)



In [4]:
# Remove rows with data anomalies
sdf_ridership_clean = (
    sdf_ridership
    .filter(
        F.col("transit_timestamp").isNotNull() &              # transit_timestamp must not be Null
        F.col("station_complex_id").isNotNull() &             # station_complex_id must not be Null
        F.col("station_complex").isNotNull() &                # station_complex must not be Null
        F.col("latitude").isNotNull() &                       # latitude must not be Null
        F.col("longitude").isNotNull() &                      # longtitude must not be Null
        (F.col("transit_timestamp") >= F.lit(START_TIME)) &   # transit time must be within specified time range
        (F.col("transit_timestamp") < F.lit(END_TIME)) &      # transit time must be within specified time range
        (F.col("ridership") >= 0) &                           # ridership must be non-negative
        (F.col("transfers") >= 0)                             # transfers must be non-negative
    )
)
n_raw = sdf_ridership.count()
n_clean = sdf_ridership_clean.count()

print(f"Raw rows:   {n_raw:,}")
print(f"Clean rows: {n_clean:,}")
print(f"Removed:    {n_raw - n_clean:,} ({(n_raw - n_clean) / n_raw:.2%})")

Raw rows:   5,140,106
Clean rows: 5,140,101
Removed:    5 (0.00%)


In [5]:
# Remove columns unused in downstream analysis
sdf_ridership_selected = (
    sdf_ridership_clean
    .select(
        F.col("transit_timestamp"),
        F.col("station_complex_id").cast("string"),
        F.col("station_complex").cast("string"),
        F.col("borough").cast("string"),
        F.col("ridership").cast("long"),
        F.col("latitude").cast("double"),
        F.col("longitude").cast("double"),
    )
)
sdf_ridership_selected.write.mode("overwrite").parquet(f"{processed_output_dir}/mta_ridership")

Specifying dataset shape:
- `sdf_ridership`, raw parquet file, contains 5140106 rows x 8 columns
- `sdf_ridership_clean`, filtered for rows with valid data, contains 5140101 rows x 8 columns 
- `sdf_ridership_selected`, filtered for columns to be used downstream, contains 5140101 rows x 7 columns

---

## Preprocess MTA Service Alerts Data
We want to remove all rows with data anomaly, such as the following: 
- alerts with NULL in any of `alert_id`, `event_id`, `date`, `affected`

In [6]:
# Read the CSV from directory (all months combined)
sdf_alerts = (
    spark.read
    .option("header", True).option("multiline", True).option("quote", '"').option("escape", '"')
    .csv("../data/raw/mta_alerts/*.csv")
    .select(
        F.col("alert_id").cast("string").alias("alert_id"),
        F.col("event_id").cast("string").alias("event_id"),
        F.col("update_number").cast("long").alias("update_number"),
        F.to_timestamp("date").alias("date"),
        F.col("agency").cast("string").alias("agency"),
        F.col("status_label").cast("string").alias("status_label"),
        F.col("affected").cast("string").alias("affected"),
        F.col("header").cast("string").alias("header"),
        F.col("description").cast("string").alias("description")
    )
)

# Print schema for reference
sdf_alerts.printSchema()

root
 |-- alert_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- update_number: long (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- agency: string (nullable = true)
 |-- status_label: string (nullable = true)
 |-- affected: string (nullable = true)
 |-- header: string (nullable = true)
 |-- description: string (nullable = true)



In [7]:
# Remove rows with data anomalies
sdf_alerts_clean = sdf_alerts.filter(
    F.col("alert_id").isNotNull() &                   # alert_id must not be Null
    F.col("event_id").isNotNull() &                   # event_id must not be Null
    F.col("date").isNotNull() &                       # date must not be Null
    F.col("affected").isNotNull() &                   # affected must not be Null
    (F.col("agency") == "NYCT Subway") &              # agency must be NYCT Subway 
    (F.col("date") >= F.lit(START_TIME)) &            # alert time must be within specified time range
    (F.col("date") < F.lit(END_TIME))                 # alert time must be within specified time range
)
n_raw = sdf_alerts.count()
n_clean = sdf_alerts_clean.count()

print(f"Raw rows:   {n_raw:,}")
print(f"Clean rows: {n_clean:,}")
print(f"Removed:    {n_raw - n_clean:,} ({(n_raw - n_clean) / n_raw:.2%})")

Raw rows:   67,502
Clean rows: 67,502
Removed:    0 (0.00%)


In [8]:
# Remove columns unused in downstream analysis
sdf_alerts_selected = (
    sdf_alerts_clean
    .select(
        F.col("event_id"),
        F.col("update_number"),
        F.col("date"),
        F.col("status_label"),
        F.col("affected"),
        F.col("header"),
        F.col("description"),
    )
)

sdf_alerts_selected.write.mode("overwrite").parquet(f"{processed_output_dir}/mta_alerts")

Specifying dataset shape:
- `sdf_alerts`, raw parquet file, contains 67502 rows x 9 columns
- `sdf_alerts_clean`, filtered for rows with valid data, contains 67502 rows x 9 columns 
- `sdf_alerts_selected`, filtered for columns to be used downstream, contains 67502 rows x 7 columns

---

## Preprocess Meteostat Weather Data
We want to remove all rows with data anomaly, such as the following: 
- weather data with NULL in any field that is used in downstream analsyis

In [9]:
# Read the CSV from directory (all months combined)
sdf_weather = (
    spark.read
    .option("header", True)
    .csv("../data/raw/meteostat_weather/*.csv")
    .select(
        F.to_timestamp("time").alias("time"),
        F.col("temp").cast("double").alias("temp"),
        F.col("dwpt").cast("double").alias("dwpt"),
        F.col("rhum").cast("double").alias("rhum"),
        F.col("prcp").cast("double").alias("prcp"),
        F.col("snow").cast("double").alias("snow"),
        F.col("wdir").cast("double").alias("wdir"),
        F.col("wspd").cast("double").alias("wspd"),
        F.col("wpgt").cast("double").alias("wpgt"),
        F.col("pres").cast("double").alias("pres"),
        F.col("tsun").cast("double").alias("tsun"),
        F.col("coco").cast("double").alias("coco")
    )
)

# Print schema for reference
sdf_weather.printSchema()

root
 |-- time: timestamp (nullable = true)
 |-- temp: double (nullable = true)
 |-- dwpt: double (nullable = true)
 |-- rhum: double (nullable = true)
 |-- prcp: double (nullable = true)
 |-- snow: double (nullable = true)
 |-- wdir: double (nullable = true)
 |-- wspd: double (nullable = true)
 |-- wpgt: double (nullable = true)
 |-- pres: double (nullable = true)
 |-- tsun: double (nullable = true)
 |-- coco: double (nullable = true)



In [10]:
# Remove rows with data anomalies
sdf_weather_clean = sdf_weather.filter(
    F.col("time").isNotNull() &                   # time must not be Null
    F.col("temp").isNotNull() &                   # temp must not be Null
    F.col("rhum").isNotNull() &                   # rhum must not be Null
    F.col("wdir").isNotNull() &                   # wdir must not be Null
    F.col("wspd").isNotNull() &                   # wspd must not be Null
    F.col("coco").isNotNull() &                   # coco must not be Null
    (F.col("time") >= F.lit(START_TIME)) &        # weather entry must be within specified time range
    (F.col("time") < F.lit(END_TIME))             # weather entry must be within specified time range
)
n_raw = sdf_weather.count()
n_clean = sdf_weather_clean.count()

print(f"Raw rows:   {n_raw:,}")
print(f"Clean rows: {n_clean:,}")
print(f"Removed:    {n_raw - n_clean:,} ({(n_raw - n_clean) / n_raw:.2%})")

Raw rows:   12,383
Clean rows: 12,383
Removed:    0 (0.00%)


In [11]:
# Remove columns unused in downstream analysis
sdf_weather_selected = (
    sdf_weather_clean
    .select(
        F.col("time"),
        F.col("coco").cast("int").alias("coco"),
    )
)

sdf_weather_selected.write.mode("overwrite").parquet(f"{processed_output_dir}/meteostat_weather")

Specifying dataset shape:
- `sdf_weather`, raw parquet file, contains 12383 rows x 12 columns
- `sdf_weather_clean`, filtered for rows with valid data, contains 12383 rows x 12 columns 
- `sdf_weather_selected`, filtered for columns to be used downstream, contains 12383 rows x 2 columns

---

## Checkpoint
By the end of this Jupyter Notebook, all the relevant datasets should have been cleaned (remove anomaly rows, remove unused columns) and saved as a parquet file or directory using PySpark.

For tlc_trips, all rows have a new combined `subtotal` column which describes the amount paid by passenger, excluding any tips (regardless of payment method).

The **processed data structures** are as follows:

tlc_trips / yellow
- `pickup_datetime`
- `dropoff_datetime`
- `PULocationID`
- `DOLocationID`
- `subtotal` = total_amount - tip_amount
- `service_type` = "yellow"

tlc_trips / green
- `pickup_datetime`
- `dropoff_datetime`
- `PULocationID`
- `DOLocationID`
- `subtotal` = total_amount - tip_amount
- `service_type` = "green"

tlc_trips / fhvhv
- `request_datetime`
- `pickup_datetime`
- `dropoff_datetime`
- `PULocationID`
- `DOLocationID`
- `subtotal` = base_passenger_fare + tolls + bcf + sales_tax + congestion_surcharge + airport_fee + cbd_congestion_fee
- `service_type` = "fhvhv"

mta_ridership
- `transit_timestamp`
- `station_complex_id`
- `station_complex`
- `borough`
- `ridership`
- `latitude`
- `longitude`

mta_alerts
- `event_id`
- `update_number`
- `date`
- `status_label`
- `affected`
- `header`
- `description`

meteostat_weather
- `time`
- `coco`

These cleaned datasets will be used for aggregation and feature engineering in the next notebook. 